# Model report
Read the questions in order. Results come only from saved files; conclusions belong to the author.

In [ ]:
ROOT_HINT = None
RUN = None
EVALUATION_ID = None
REFERENCE_RUN = None
REFERENCE_EVALUATION_ID = None

In [ ]:
import os, sys
from pathlib import Path
os.environ.update(CUDA_VISIBLE_DEVICES='', HIP_VISIBLE_DEVICES='', MPLBACKEND='Agg')
candidates = ([Path(ROOT_HINT)] if ROOT_HINT else []) + [Path.cwd(), *Path.cwd().parents]
ROOT = next(p for p in candidates if (p/'cvdl/reporting.py').is_file())
sys.path.insert(0, str(ROOT))
from cvdl.reporting import (SavedReport, cpu_offline_guard, compatible, headline, run_card,
    training_plots, panel_plot, miss_plots, hard_plots, comparison_plot, source_lines, appendix)
cpu_offline_guard()
report = SavedReport(ROOT, RUN, EVALUATION_ID)
reference = SavedReport(ROOT, REFERENCE_RUN, REFERENCE_EVALUATION_ID) if REFERENCE_RUN else None
if reference: compatible(report, reference)
assert 'torch' not in sys.modules

## 0. How good is this model, in short?
A **false alarm** calls a clean image a hidden message. A **miss** fails to detect a hidden message. **Overall error** is the average of those two rates. **AUC** measures how often a hidden-message image scores above a clean image (ties count half); it runs from 0 to 1.

In [ ]:
headline(report)

## 1. What was trained?

In [ ]:
run_card(report, reference)

## 2. Did training go well?
These are the saved checkpoint-selection curves, not the later threshold-calibration results. The chart names the original validation.

In [ ]:
training_plots(report)

## 3. Where does it fail?
All bars use this model’s one saved validation cut-off.

In [ ]:
panel_plot(report)

## 4. Which hidden messages does it miss?
**k** is embedding strength: the number of low-order bits available for hiding a message. **bpp** is the requested message size in bits per pixel. Grid bins are 0.05–0.10, 0.10–0.15, 0.15–0.35 and 0.35–0.50 bpp (last upper edge included). Higher-strength results, if saved, are in the appendix.

In [ ]:
miss_tables = miss_plots(report)

## 5. Does it break on hard images?
“Small message” means 0.05–0.15 bpp; “noisy” uses the frozen high-noise selection. Error bars are saved Wilson 95% ranges for tiles/pairs, not ranges grouped by photo. Cells can share images.

In [ ]:
hard_plots(report)

## 6. Better or worse than the reference?
Values are percentage-point differences, not relative percentages. Each model keeps its own saved validation cut-off. Only identical frozen test recipes are compared.

In [ ]:
comparison_plot(report, reference)

## 7. Would knowing the source help?
Reference only: source-specific cut-offs are not the deployment policy. There is no DIV2K or mixed-source reference. ALASKA shares one source-specific cut-off across both camera groups.

In [ ]:
source_lines(report)

## 8. Author notes
- **What changed?** [Pending: author]
- **What do the saved results show?** [Pending: cite charts or appendix rows]
- **What remains weak?** [Pending: author]
- **How does the reference compare?** [Pending / not supplied]
- **Limits:** [Pending: source mix, image sizes, shared photos, changed test protocols]
- **Next decision:** [Pending: author]

No interpretation is generated. Old 512-pixel / k1–8 results are not like-for-like with the balanced 256-pixel / k1–4 suite.

## Appendix — lookup only
Rates are rounded to one decimal, AUC and cut-offs to three decimals. Test-optimised and fixed-0.500 results are diagnostics, not deployment results. Saved files keep full precision.

In [ ]:
appendix(report, reference)

In [ ]:
assert os.environ['CUDA_VISIBLE_DEVICES'] == '' and os.environ['HIP_VISIBLE_DEVICES'] == ''
assert 'torch' not in sys.modules
import socket
try:
    socket.getaddrinfo('external-network.invalid', 443)
except PermissionError:
    pass
else:
    raise AssertionError('external network guard missing')